# Image ↔ RNA cluster confusion matrix (RNA-defined clusters)


- **clusters are defined on the RNA side** from the LLM-annotated clustering in
  `cropseq_clusters_llm_annotated.csv` (its per-cluster `genes` lists give the
  gene → cluster assignment plus the `llm_annotation` / `llm_confidence` etc.),
  restricted to clusters with `llm_confidence >= CONFIDENCE_MIN`. NOTE: the
  `leiden_r30` baked into `cropseq_svaeplus_embedding_phate.h5ad` is a *different*
  clustering run and is intentionally not used for cluster membership.
- the **image leiden resolution is swept** to find the best alignment (ARI)
- confusion matrix rows are RNA clusters annotated with their dominant
  `llm_annotation`, columns are image clusters
- the full analysis is run for both `CONFIDENCE_MIN = 4` and `CONFIDENCE_MIN = 5`


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import linkage, leaves_list, optimal_leaf_ordering
from scipy.spatial.distance import pdist
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150
plt.rcParams["pdf.fonttype"] = 42  # embed fonts in PDF for Illustrator compatibility
plt.rcParams["svg.fonttype"] = 'none'  # keep text as text in SVG for Illustrator

# **Before public release**: all inputs live in the central dataset under ../../data/
# (a symlink to the curated dataset; see README). These paths are the only lines
# that change when the dataset is released publicly.
IMAGE_PATH = '../../data/ops_gene_embedding_phate_phase_only.h5ad'             # OPS phase-only gene embedding (PHATE + leiden + neighbour graph)
RNA_PATH   = '../../data/cropseq_svaeplus_embedding_phate.h5ad'                # crop-seq sVAEplus gene embedding (PHATE + leiden + neighbour graph)
LLM_CSV    = '../../data/figures/figure_5/cropseq_clusters_llm_annotated.csv'  # cluster-level LLM annotations + gene→cluster membership

OUTPUT_DIR = Path('../../output/figure_5')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNA_LEIDEN_COL     = 'leiden_r30'         # RNA cluster column; rebuilt from the CSV gene lists in the next cell (the h5ad's own leiden_r30 is a different run)
RESOLUTION_GRID    = np.linspace(25, 35, 11)  # image-side leiden resolutions to sweep
CONFIDENCE_CUTOFFS = [4, 5]               # filter RNA clusters by llm_confidence >= cutoff
MIN_OVERLAP        = 2
LEIDEN_SEED        = 1

In [ ]:
img = ad.read_h5ad(IMAGE_PATH)
rna = ad.read_h5ad(RNA_PATH)
print('image:', img.shape, '|  RNA:', rna.shape)
assert 'Gene name' in rna.obs.columns, 'RNA obs missing "Gene name"'

# Cluster-level LLM annotations come from the central CSV.
#
# IMPORTANT: the leiden_r30 baked into this h5ad is a *different* clustering run
# than the one the LLM annotations were built on — only ~26% of per-gene cluster
# memberships agree — so merging on `cluster_id == leiden_r30` would attach the
# wrong annotation to most genes. Instead we rebuild RNA_LEIDEN_COL from the
# CSV's authoritative per-cluster gene lists (gene membership is invariant) and
# use that as the RNA cluster definition downstream. This reproduces the original
# clustering the annotations describe.
llm = pd.read_csv(LLM_CSV)
# cluster_id in the CSV is a string ('0','1',...); keep it as such.
llm['cluster_id'] = llm['cluster_id'].astype(str)

# gene -> cluster_id from the CSV's semicolon-separated gene lists.
gene_to_cluster = {}
for _, r in llm.iterrows():
    for g in str(r['genes']).split(';'):
        gene_to_cluster[g.strip()] = str(r['cluster_id'])
rna.obs[RNA_LEIDEN_COL] = rna.obs['Gene name'].astype(str).map(gene_to_cluster)
_unmapped = rna.obs[RNA_LEIDEN_COL].isna().sum()
assert _unmapped == 0, f'{_unmapped} genes not found in any CSV cluster gene list'
rna.obs[RNA_LEIDEN_COL] = rna.obs[RNA_LEIDEN_COL].astype(str)

cluster_to = {col: dict(zip(llm['cluster_id'], llm[col]))
              for col in ['coherent_annotation', 'confidence', 'notes',
                          'dominant_chrom_arm', 'n_genes']}

rna.obs['llm_annotation'] = rna.obs[RNA_LEIDEN_COL].map(cluster_to['coherent_annotation'])
rna.obs['llm_confidence'] = rna.obs[RNA_LEIDEN_COL].map(cluster_to['confidence']).astype('Int64')
rna.obs['llm_notes']      = rna.obs[RNA_LEIDEN_COL].map(cluster_to['notes'])
rna.obs['llm_dominant_chrom_arm'] = rna.obs[RNA_LEIDEN_COL].map(cluster_to['dominant_chrom_arm'])
rna.obs['llm_cluster_size']       = rna.obs[RNA_LEIDEN_COL].map(cluster_to['n_genes']).astype('Int64')
# A short label suitable for plot legends: "<annotation> (conf=N)".
rna.obs['llm_label'] = (
    rna.obs['llm_annotation'].fillna('—').astype(str)
    + ' (conf=' + rna.obs['llm_confidence'].astype(str) + ')'
)
print('coverage:', rna.obs['llm_annotation'].notna().sum(), '/', rna.n_obs)
print('confidence distribution (per gene):')
print(rna.obs['llm_confidence'].value_counts(dropna=False).sort_index().to_string())

assert 'llm_annotation' in rna.obs.columns
assert 'llm_confidence' in rna.obs.columns

# Gene symbol lives in obs.index on the image side, in obs['Gene name'] on the RNA side.
img.obs_names = img.obs_names.astype(str)
rna_gene = rna.obs['Gene name'].astype(str)
rna = rna[~rna_gene.duplicated()].copy()
rna.obs_names = rna_gene[~rna_gene.duplicated()].values

common_all = sorted(set(img.obs_names) & set(rna.obs_names))
print(f'common genes (all): {len(common_all)}')

# llm_annotation / llm_confidence are cluster-level on the RNA side (constant
# within each RNA cluster). Keep two per-gene Series for downstream use:
#   llm_conf_per_gene: confidence carried by each gene (== its cluster's confidence)
#   llm_ann_per_gene : annotation string carried by each gene
llm_conf_per_gene = pd.to_numeric(rna.obs['llm_confidence'], errors='coerce')
llm_ann_per_gene  = rna.obs['llm_annotation'].astype(str)
llm_conf_per_gene.index = rna.obs_names
llm_ann_per_gene.index  = rna.obs_names

# Sanity: confidence/annotation should be constant per RNA cluster.
_rna_clu = rna.obs[RNA_LEIDEN_COL].astype(str)
_rna_clu.index = rna.obs_names
n_conf_unique = pd.DataFrame({'c': _rna_clu, 'v': llm_conf_per_gene}).groupby('c')['v'].nunique().max()
n_ann_unique  = pd.DataFrame({'c': _rna_clu, 'v': llm_ann_per_gene }).groupby('c')['v'].nunique().max()
assert n_conf_unique == 1 and n_ann_unique == 1, 'expected cluster-level llm annotations'
print(f'RNA clusters at {RNA_LEIDEN_COL}: {_rna_clu.nunique()} total')
for cut in CONFIDENCE_CUTOFFS:
    n_kept = (llm_conf_per_gene[~llm_conf_per_gene.isna()] >= cut).groupby(
        _rna_clu.loc[llm_conf_per_gene.dropna().index]
    ).any().sum()
    n_genes = int((llm_conf_per_gene.reindex(common_all) >= cut).sum())
    print(f'  llm_confidence >= {cut}: {n_kept} clusters / {n_genes} common-set genes')

In [ ]:
rna.obs.columns.tolist()

In [ ]:
def confusion_iou(rna_labels, img_labels, min_overlap=1):
    """Counts table + Hungarian-permuted IoU table.

    Rows = RNA cluster, cols = image cluster. Plain layout: Hungarian-matched
    rows/cols at positions 0..n_match-1 on the diagonal; unmatched rows/cols
    appended in numeric order. Used for metric calculation only — see
    `arrange_columns_for_display` for the heatmap layout.
    """
    ct = pd.crosstab(pd.Series(rna_labels, name='rna'),
                     pd.Series(img_labels, name='image'))
    def numkey(idx):
        return pd.to_numeric(idx, errors='coerce').fillna(np.inf)
    ct = ct.iloc[np.argsort(numkey(ct.index)), np.argsort(numkey(ct.columns))]

    keep_rows = ct.sum(axis=1) >= min_overlap
    keep_cols = ct.sum(axis=0) >= min_overlap
    ct = ct.loc[keep_rows, keep_cols]
    if ct.shape[0] == 0 or ct.shape[1] == 0:
        return ct, pd.DataFrame(index=ct.index, columns=ct.columns, dtype=float)

    row_sizes = ct.sum(axis=1).values
    col_sizes = ct.sum(axis=0).values
    inter = ct.values
    union = row_sizes[:, None] + col_sizes[None, :] - inter
    iou = np.where(union > 0, inter / union, 0.0)

    row_match, col_match = linear_sum_assignment(-iou)
    row_order = list(row_match) + [i for i in range(iou.shape[0]) if i not in set(row_match)]
    col_order = list(col_match) + [j for j in range(iou.shape[1]) if j not in set(col_match)]
    ct_re  = ct.iloc[row_order, col_order]
    iou_re = pd.DataFrame(iou, index=ct.index, columns=ct.columns).iloc[row_order, col_order]
    return ct_re, iou_re

def seriate_pairs(ct, iou):
    """Reorder the Hungarian-paired block so off-diagonal mass clusters near the diagonal.

    `ct` / `iou` come from `confusion_iou`, where rows 0..n_match-1 are
    Hungarian-paired with cols 0..n_match-1. We permute those n_match indices
    *identically* on both axes (so the diagonal Hungarian matches stay on the
    diagonal) using hierarchical clustering with optimal leaf ordering on the
    symmetrized off-diagonal mass. Extras (rows or cols beyond n_match) keep
    their numeric order — unmatched columns get re-placed downstream by
    `arrange_columns_for_display`.
    """
    n_match = min(ct.shape[0], ct.shape[1])
    if n_match < 3:
        return ct, iou

    sub = ct.iloc[:n_match, :n_match].values.astype(float)
    # Symmetrize so distance reflects bidirectional off-diagonal flow, then
    # zero the diagonal so the (large) Hungarian matches don't dominate.
    M = sub + sub.T
    np.fill_diagonal(M, 0.0)
    # Cosine is undefined for all-zero rows — give those a tiny uniform vector.
    zero_rows = M.sum(axis=1) == 0
    if zero_rows.any():
        M = M.copy()
        M[zero_rows] = 1e-9
    try:
        D = pdist(M, metric='cosine')
        Z = linkage(D, method='average')
        Z = optimal_leaf_ordering(Z, D)
        order = list(leaves_list(Z))
    except Exception:
        return ct, iou

    row_order = order + list(range(n_match, ct.shape[0]))
    col_order = order + list(range(n_match, ct.shape[1]))
    return ct.iloc[row_order, col_order], iou.iloc[row_order, col_order]

def arrange_columns_for_display(ct, iou):
    """Rearrange columns of a plain-layout confusion matrix for prettier display.

    Diagonal Hungarian pairs stay in place. Each unmatched image column is
    moved next to the RNA row where it has its maximum IoU; multiple extras at
    the same row are sorted strongest-first so the closest extras sit nearest
    the diagonal. Columns no row owns (all-zero IoU) end up at the right.
    Does NOT touch the row order or the metric on the diagonal.
    """
    Ni, Nj = iou.shape
    n_match = min(Ni, Nj)
    iou_vals = iou.values

    matched_col_pos = list(range(n_match))
    unmatched_col_pos = list(range(n_match, Nj))
    col_argmax_row = {c: int(np.argmax(iou_vals[:, c])) for c in unmatched_col_pos}

    placed = set()
    new_col_pos = []
    for row_pos in range(Ni):
        if row_pos < n_match:
            new_col_pos.append(row_pos)
            placed.add(row_pos)
        extras = [
            c for c in unmatched_col_pos
            if c not in placed
            and iou_vals[:, c].max() > 0
            and col_argmax_row[c] == row_pos
        ]
        extras.sort(key=lambda c: -iou_vals[row_pos, c])
        new_col_pos.extend(extras)
        placed.update(extras)
    new_col_pos.extend([c for c in range(Nj) if c not in placed])

    return ct.iloc[:, new_col_pos], iou.iloc[:, new_col_pos]

def annotate_rna_clusters(ct, gene_clusters, gene_annotation):
    """Replace `ct.index` (RNA cluster IDs) with '<id>: <dominant llm_annotation>'."""
    df = pd.DataFrame({'cluster': gene_clusters.astype(str),
                       'ann':     gene_annotation.astype(str)}).dropna()
    mode_per_cluster = (df.groupby('cluster')['ann']
                         .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else '—'))
    ct = ct.copy()
    ct.index = [f'{c}: {mode_per_cluster.get(c, "—")}' for c in ct.index]
    return ct

OTHER_COL = 'other (img)'
OTHER_ROW = 'other (RNA)'
# Sentinel used only for the corner cell (the "dropped × dropped" intersection
# has no single-cluster fraction interpretation, so it stays grey via
# cmap.set_under in plot_confusion). All other catch-all cells get a real
# 0..1 coverage fraction and colour with the main Blues cmap.
OTHER_IOU = -0.01

def add_catch_all(ct_disp, iou_disp, kept_rna_ids, kept_img_ids, ct_full):
    """Append a catch-all column and row to a display matrix.

    Each catch-all cell shows the number of genes that fell into the dropped
    portion of the other axis. Cell colour is a coverage fraction on the same
    0..1 (0..100 %) scale as the main matrix's IoU:
      - catch-all column at row R : count / row_total
        = fraction of RNA cluster R that landed in dropped image clusters
      - catch-all row at col C    : count / col_total
        = fraction of image cluster C that landed in dropped RNA clusters
    The corner cell counts genes in BOTH dropped axes; it has no single-
    cluster denominator, so it is left grey (OTHER_IOU sentinel rendered by
    cmap.set_under in plot_confusion).

    Args:
      ct_disp/iou_disp : final display layout (rows annotated, columns permuted)
      kept_rna_ids     : original RNA cluster IDs in `ct_disp` row order
      kept_img_ids     : original image cluster IDs in `ct_disp` column order
      ct_full          : full (unfiltered) crosstab of rna × img on the same
                         gene subset that produced ct_disp
    """
    ct_full_s = ct_full.copy()
    ct_full_s.index   = ct_full_s.index.astype(str)
    ct_full_s.columns = ct_full_s.columns.astype(str)
    kept_rna_set = set(map(str, kept_rna_ids))
    kept_img_set = set(map(str, kept_img_ids))
    dropped_img  = [c for c in ct_full_s.columns if c not in kept_img_set]
    dropped_rna  = [r for r in ct_full_s.index   if r not in kept_rna_set]

    if dropped_img:
        catch_col = ct_full_s.reindex(index=list(map(str, kept_rna_ids)),
                                      columns=dropped_img).fillna(0).sum(axis=1)
    else:
        catch_col = pd.Series(0, index=list(map(str, kept_rna_ids)), dtype=int)
    if dropped_rna:
        catch_row = ct_full_s.reindex(index=dropped_rna,
                                      columns=list(map(str, kept_img_ids))).fillna(0).sum(axis=0)
    else:
        catch_row = pd.Series(0, index=list(map(str, kept_img_ids)), dtype=int)
    corner = int(ct_full_s.reindex(index=dropped_rna, columns=dropped_img).fillna(0).sum().sum()) \
        if (dropped_img and dropped_rna) else 0

    # Counts: append catch-all column, then catch-all row + corner.
    ct2  = ct_disp.copy()
    iou2 = iou_disp.copy()
    catch_col_counts = [int(catch_col.loc[str(_id)]) for _id in kept_rna_ids]
    catch_row_counts = [int(catch_row.loc[str(_id)]) for _id in kept_img_ids]
    ct2[OTHER_COL]     = catch_col_counts
    ct2.loc[OTHER_ROW] = catch_row_counts + [corner]
    ct2 = ct2.astype(int)

    # Coverage fractions (0..1) — each row/col total now includes the catch-all
    # entry, so the fraction = catch-all count / cluster size on the common set.
    row_totals = ct2.drop(OTHER_ROW).sum(axis=1)          # one per kept RNA row
    col_totals = ct2.drop(columns=OTHER_COL).sum(axis=0)  # one per kept image col
    catch_col_frac = (pd.Series(catch_col_counts, index=ct2.index.drop(OTHER_ROW))
                        / row_totals.replace(0, np.nan)).fillna(0.0).clip(0, 1)
    catch_row_frac = (pd.Series(catch_row_counts, index=[c for c in ct2.columns if c != OTHER_COL])
                        / col_totals.replace(0, np.nan)).fillna(0.0).clip(0, 1)
    iou2[OTHER_COL]     = catch_col_frac.values                   # kept RNA rows
    iou2.loc[OTHER_ROW] = list(catch_row_frac.values) + [OTHER_IOU]  # corner stays grey
    return ct2, iou2

def plot_confusion(ct, iou, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(5, 0.35 * ct.shape[1] + 2),
                                        max(4, 0.30 * ct.shape[0] + 2)))
    cmap = plt.get_cmap('Blues').copy()
    cmap.set_under('lightgrey')  # only the corner cell (OTHER_IOU < 0) renders grey
    sns.heatmap(
        iou * 100, annot=ct.values, fmt='d', cmap=cmap,
        vmin=0, vmax=100,
        xticklabels=ct.columns, yticklabels=ct.index,
        annot_kws={'size': 10}, linewidths=0.2, linecolor='lightgrey',
        cbar_kws={'label': 'IoU % (main)  ·  coverage % missing (catch-all)'}, ax=ax,
    )
    # Thicker separator before the catch-all column / above the catch-all row.
    if OTHER_COL in ct.columns:
        x_sep = list(ct.columns).index(OTHER_COL)
        ax.axvline(x_sep, color='black', lw=1.0)
    if OTHER_ROW in ct.index:
        y_sep = list(ct.index).index(OTHER_ROW)
        ax.axhline(y_sep, color='black', lw=1.0)
    ax.set_xlabel('image cluster'); ax.set_ylabel('RNA cluster: llm_annotation')
    ax.set_title(title, fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)
    return ax

In [ ]:
# Resolution sweep on the IMAGE side: at each candidate resolution, compute
# leiden on the (pre-existing) image neighbour graph and score against the
# (fixed) high-confidence RNA leiden_r30 clusters using ARI on the common
# high-confidence gene set. Pick the highest-ARI resolution.
import scib as _scib_for_sweep

# Image already has a neighbour graph baked in (obsp/connectivities, obsp/distances,
# uns/neighbors); sc.tl.leiden picks it up directly.

def _ari(labels_a, labels_b):
    tmp = ad.AnnData(
        X=np.empty((len(labels_a), 0), dtype=np.float32),
        obs=pd.DataFrame({
            'a': pd.Categorical(labels_a),
            'b': pd.Categorical(labels_b),
        }),
    )
    return float(_scib_for_sweep.metrics.ari(tmp, 'a', 'b'))

def _matched_diag_iou(rna_labels, img_labels, min_overlap):
    ct, iou = confusion_iou(rna_labels, img_labels, min_overlap=min_overlap)
    if ct.shape[0] == 0 or ct.shape[1] == 0:
        return 0.0, 0.0, 0, 0
    nm = min(ct.shape[0], ct.shape[1])
    diag = np.diag(iou.values)[:nm]
    return float(diag.mean()), float(diag.sum()), int(ct.shape[0]), int(ct.shape[1])

def resolution_sweep(common_genes):
    rows = []
    rna_labels_common = rna[common_genes].obs[RNA_LEIDEN_COL].astype(str).values
    for r in RESOLUTION_GRID:
        key = f'leiden_r{int(r) if float(r).is_integer() else r}_img'
        sc.tl.leiden(
            img, resolution=r, flavor='igraph', directed=False, n_iterations=2,
            random_state=LEIDEN_SEED, key_added=key,
        )
        img_labels_common = img[common_genes].obs[key].astype(str).values
        ari_val = _ari(rna_labels_common, img_labels_common)
        mean_iou, sum_iou, n_rna_kept, n_img_kept = _matched_diag_iou(
            rna_labels_common, img_labels_common, MIN_OVERLAP)
        rows.append({'resolution': r, 'key': key,
                     'ari': ari_val,
                     'mean_diag_iou': mean_iou, 'sum_diag_iou': sum_iou,
                     'n_rna_kept': n_rna_kept, 'n_img_kept': n_img_kept,
                     'n_img_clusters': int(img.obs[key].nunique())})
    return pd.DataFrame(rows).sort_values('resolution').reset_index(drop=True)

# Run the sweep once per confidence cutoff. Each cutoff defines a different
# common-gene subset, which in turn picks a different best image resolution.
sweeps = {}
commons = {}
best_keys = {}
for cut in CONFIDENCE_CUTOFFS:
    keep_genes = sorted(set(common_all) & set(
        llm_conf_per_gene.index[llm_conf_per_gene >= cut].astype(str)
    ))
    commons[cut] = keep_genes
    print(f'\n=== confidence >= {cut}:  {len(keep_genes)} common-set genes ===')
    sw = resolution_sweep(keep_genes)
    print(sw.to_string(index=False))
    best_idx = sw['ari'].idxmax()
    sweeps[cut] = sw
    best_keys[cut] = sw.loc[best_idx, 'key']
    print(f"  best image resolution (by ARI): r={sw.loc[best_idx,'resolution']}"
          f"  ARI={sw.loc[best_idx,'ari']:.3f}  mean_diag_iou={sw.loc[best_idx,'mean_diag_iou']:.3f}"
          f"  ({sw.loc[best_idx,'n_img_clusters']} img clusters)")

In [ ]:
# Final confusion matrix at RNA leiden_r30 vs the swept-best image leiden,
# one panel per confidence cutoff. Rows = RNA clusters (annotated with their
# dominant llm_annotation), cols = image clusters. A catch-all column (right)
# and row (bottom), drawn in grey, count the genes that fell into clusters the
# MIN_OVERLAP filter dropped — so each row/column total now matches the kept
# cluster's actual size.
results = {}

for cut in CONFIDENCE_CUTOFFS:
    common = commons[cut]
    img_key = best_keys[cut]

    rna_c = rna[common].copy()
    img_c = img[common].copy()
    rna_labels = rna_c.obs[RNA_LEIDEN_COL].astype(str).values
    img_labels = img_c.obs[img_key].astype(str).values

    ct, iou = confusion_iou(rna_labels, img_labels, min_overlap=MIN_OVERLAP)
    # Seriate the Hungarian-paired block: permute paired rows and cols
    # together (diagonal preserved) so off-diagonal mass clusters near it.
    ct, iou = seriate_pairs(ct, iou)

    gene_cluster = pd.Series(rna_labels, index=common)
    gene_ann     = llm_ann_per_gene.reindex(common)
    ct_ann = annotate_rna_clusters(ct, gene_cluster, gene_ann)
    iou_ann = iou.copy(); iou_ann.index = ct_ann.index

    ct_disp, iou_disp = arrange_columns_for_display(ct_ann, iou_ann)

    # Full (unfiltered) crosstab on the same gene subset, used to populate the
    # catch-all row/column. `kept_*_ids` carry the original cluster IDs in the
    # final display order (ct's row order is preserved through annotate / arrange;
    # ct_disp.columns are the original IDs after permutation).
    ct_full = pd.crosstab(pd.Series(rna_labels).astype(str),
                          pd.Series(img_labels).astype(str))
    kept_rna_ids = list(ct.index.astype(str))               # row order
    kept_img_ids = list(ct_disp.columns.astype(str))        # column order (permuted)
    ct_disp, iou_disp = add_catch_all(ct_disp, iou_disp, kept_rna_ids, kept_img_ids, ct_full)

    n_rna_kept = ct.shape[0]
    n_img_kept = ct.shape[1]
    n_rna_total = rna_c.obs[RNA_LEIDEN_COL].nunique()
    n_img_total = img_c.obs[img_key].nunique()
    nm = min(n_rna_kept, n_img_kept)
    diag = np.diag(iou.values)[:nm]
    # Catch-all stats: how much of each kept cluster falls outside the displayed matrix.
    catch_col_vals = ct_disp[OTHER_COL].drop(OTHER_ROW).values
    catch_row_vals = ct_disp.loc[OTHER_ROW].drop(OTHER_COL).values
    row_totals = ct_disp.drop(OTHER_ROW).sum(axis=1).values
    col_totals = ct_disp.drop(columns=OTHER_COL).sum(axis=0).values
    pct_other_row = 100 * catch_col_vals.sum() / max(row_totals.sum(), 1)
    pct_other_col = 100 * catch_row_vals.sum() / max(col_totals.sum(), 1)
    print(f'\n=== confidence >= {cut} ===')
    print(f'After min_overlap>={MIN_OVERLAP} filter:')
    print(f'  RNA clusters kept   : {n_rna_kept} / {n_rna_total} total')
    print(f'  image clusters kept : {n_img_kept} / {n_img_total} total')
    print(f'  mean diag IoU = {diag.mean():.3f}, sum = {diag.sum():.2f}')
    print(f'  catch-all column: {int(catch_col_vals.sum())} genes '
          f'({pct_other_row:.1f}% of kept RNA-row totals) sit in dropped image clusters')
    print(f'  catch-all row   : {int(catch_row_vals.sum())} genes '
          f'({pct_other_col:.1f}% of kept image-col totals) sit in dropped RNA clusters')

    title = (f'rna {RNA_LEIDEN_COL} (conf>={cut})  vs  image {img_key}  '
             f'(rna {n_rna_kept}/{n_rna_total} × image {n_img_kept}/{n_img_total},  '
             f'{len(common)} high-confidence genes;  '
             f'mean diag IoU = {diag.mean():.3f}, sum = {diag.sum():.2f})')
    plot_confusion(ct_disp, iou_disp, title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_{RNA_LEIDEN_COL}_vs_{img_key}'
                             f'_confidence{cut}.pdf', bbox_inches='tight')
    plt.show()

    results[cut] = dict(
        common=common, img_key=img_key,
        ct=ct, iou=iou, ct_ann=ct_ann, iou_ann=iou_ann,
        ct_disp=ct_disp, iou_disp=iou_disp, ct_full=ct_full,
        observed_mean=float(diag.mean()),
        rna_labels=rna_labels, img_labels=img_labels,
    )

In [ ]:
# RNA clusters with llm_confidence >= 5, ranked by IoU vs their Hungarian-
# matched image cluster (best → worst). Pulls from results[5]: rows of `ct`
# are RNA clusters and the first n_match cols are Hungarian-paired image
# clusters; seriate_pairs preserves that pairing on the diagonal, so
# diag(iou) gives each cluster's match strength.
CONF_RANK_CUT = 5
res = results[CONF_RANK_CUT]
ct      = res['ct']        # RNA × image, Hungarian-paired diagonal preserved
iou     = res['iou']
ct_full = res['ct_full'].copy()
ct_full.index   = ct_full.index.astype(str)
ct_full.columns = ct_full.columns.astype(str)
n_match = min(ct.shape[0], ct.shape[1])

# Dominant llm_annotation per RNA cluster. `res['rna_labels']` is per-gene and
# aligned with `res['common']`; we anchor both Series on the gene-name index
# so they line up in the DataFrame (otherwise pandas index-aligns a 0..N
# integer index against gene names and silently drops every row).
_cluster_s = pd.Series(res['rna_labels'], index=res['common']).astype(str)
_ann_s     = llm_ann_per_gene.reindex(res['common']).astype(str)
_df = pd.DataFrame({'cluster': _cluster_s, 'ann': _ann_s}).dropna()
_rna_mode = (_df.groupby('cluster')['ann']
                .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else '—'))

rows = []
for i in range(n_match):
    rna_id = str(ct.index[i])
    img_id = str(ct.columns[i])
    rows.append({
        'rna_id':         rna_id,
        'llm_annotation': _rna_mode.get(rna_id, '—'),
        'img_id':         img_id,
        'IoU':            round(float(iou.iloc[i, i]), 3),
        'n_overlap':      int(ct.iloc[i, i]),
        'rna_size':       int(ct_full.loc[rna_id].sum()),
        'img_size':       int(ct_full[img_id].sum()),
    })

ranked = pd.DataFrame(rows).sort_values('IoU', ascending=False).reset_index(drop=True)
print(f'RNA clusters with llm_confidence >= {CONF_RANK_CUT}, '
      f'ranked by IoU vs Hungarian-matched image cluster')
print(f'(image side = {res["img_key"]}, {len(ranked)} RNA clusters; '
      f'sizes on the high-confidence common-gene set)\n')
with pd.option_context('display.max_rows', None,
                       'display.max_colwidth', 80,
                       'display.width', 200):
    print(ranked.to_string(index=False))

# Save for downstream use.
out_path = (OUTPUT_DIR / f'confusion_{RNA_LEIDEN_COL}_vs_{res["img_key"]}'
                         f'_confidence{CONF_RANK_CUT}_rna_ranked.csv')
ranked.to_csv(out_path, index=False)
print(f'\nsaved: {out_path}')

In [ ]:
# PHATE embeddings highlighting kept RNA / image clusters, one figure per cutoff.
# Each Hungarian-matched image cluster inherits the colour of its paired RNA
# cluster so corresponding clusters look the same across panels.
import matplotlib.colors as mcolors
import matplotlib.cm as cm

RNA_PHATE_KEY = next((k for k in rna.obsm.keys() if k.startswith('X_PHATE')), None)
# assert 'X_phate' in img.obsm, 'image AnnData missing X_phate'
IMG_PHATE_KEY = next((k for k in img.obsm.keys() if k.upper().startswith('X_PHATE')), None)
assert RNA_PHATE_KEY is not None, f'no X_PHATE_* key in rna.obsm; got {list(rna.obsm)}'
assert IMG_PHATE_KEY is not None, f'no X_PHATE_* key in img.obsm; got {list(img.obsm)}'

def build_palette(n):
    chunks = ['tab20', 'tab20b', 'tab20c', 'Set3', 'Pastel1', 'Pastel2', 'Accent']
    colors = []
    for name in chunks:
        for c in plt.get_cmap(name).colors:
            colors.append(mcolors.to_hex(c))
            if len(colors) >= n:
                return colors
    while len(colors) < n:
        colors.append(mcolors.to_hex(cm.get_cmap('hsv')((len(colors) % 60) / 60)))
    return colors

for cut in CONFIDENCE_CUTOFFS:
    res = results[cut]
    common = res['common']; img_key = res['img_key']
    ct = res['ct']; ct_ann = res['ct_ann']

    kept_rna_ids = list(ct.index.astype(str))      # ordered as in confusion matrix
    kept_img_ids = list(ct.columns.astype(str))
    rna_labels_ord = list(ct_ann.index)            # "id: annotation"
    rna_id_to_label = dict(zip(kept_rna_ids, rna_labels_ord))

    n_match = min(len(kept_rna_ids), len(kept_img_ids))
    n_extra_img = max(0, len(kept_img_ids) - n_match)
    colors_all = build_palette(len(kept_rna_ids) + n_extra_img)

    rna_palette = {lbl: colors_all[i] for i, lbl in enumerate(rna_labels_ord)}
    img_palette = {}
    for k in range(n_match):
        img_palette[kept_img_ids[k]] = rna_palette[rna_labels_ord[k]]
    for k in range(n_extra_img):
        img_palette[kept_img_ids[n_match + k]] = colors_all[len(kept_rna_ids) + k]

    common_set = set(common)
    rna_clu = rna.obs[RNA_LEIDEN_COL].astype(str).values
    rna_in_common = np.array([g in common_set for g in rna.obs_names])
    rna.obs['phate_highlight'] = pd.Categorical([
        rna_id_to_label[c] if (c in rna_id_to_label and ok) else np.nan
        for c, ok in zip(rna_clu, rna_in_common)
    ])

    img_clu = img.obs[img_key].astype(str).values
    img_in_common = np.array([g in common_set for g in img.obs_names])
    kept_img_set = set(kept_img_ids)
    img.obs['phate_highlight'] = pd.Categorical([
        c if (c in kept_img_set and ok) else np.nan
        for c, ok in zip(img_clu, img_in_common)
    ])

    print(f'\n=== confidence >= {cut} ===')
    print(f'  RNA highlighted genes : {rna.obs["phate_highlight"].notna().sum()}'
          f' / {rna.n_obs}  ({len(kept_rna_ids)} clusters)')
    print(f'  image highlighted genes: {img.obs["phate_highlight"].notna().sum()}'
          f' / {img.n_obs}  ({len(kept_img_ids)} clusters)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sc.pl.embedding(
        rna, basis=RNA_PHATE_KEY, color='phate_highlight', ax=axes[0],
        show=False, frameon=False, na_color='lightgrey', palette=rna_palette,
        legend_fontsize=5, legend_loc='on data', legend_fontoutline=True,
        title=f'RNA PHATE — kept {RNA_LEIDEN_COL} clusters (conf>={cut}) '
              f'({len(kept_rna_ids)} of {rna.obs[RNA_LEIDEN_COL].nunique()})',
    )
    sc.pl.embedding(
        img, basis=IMG_PHATE_KEY, color='phate_highlight', ax=axes[1],
        show=False, frameon=False, na_color='lightgrey', palette=img_palette,
        legend_fontsize=5, legend_loc='right margin',
        title=f'image PHATE — kept {img_key} clusters '
              f'({len(kept_img_ids)} of {img.obs[img_key].nunique()})',
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'phate_highlighted_confidence{cut}.pdf', bbox_inches='tight')
    plt.show()

## Sanity check

**ARI permutation null**: shuffle the per-gene image cluster labels (preserving cluster sizes), re-compute the Adjusted Rand Index between the RNA and image partitions, repeat 1000×. ARI is a partition-similarity metric, so it doesn't depend on Hungarian matching or cluster labelling. If the null distribution sits well below the observed ARI, the RNA ↔ image alignment is real rather than an artifact of the alignment / filtering machinery.

In [ ]:
# Permutation null on ARI (Adjusted Rand Index) via scib.metrics.ari.
# Partition-similarity metric — doesn't depend on labelling or Hungarian matching.
import scib

N_PERMUTATIONS_ARI = 1000

for cut in CONFIDENCE_CUTOFFS:
    res = results[cut]
    rng = np.random.default_rng(LEIDEN_SEED)
    common = res['common']
    rna_labels_obs = res['rna_labels']
    img_labels_obs = res['img_labels']

    tmp = ad.AnnData(
        X=np.empty((len(common), 0), dtype=np.float32),
        obs=pd.DataFrame({'rna_cluster': rna_labels_obs,
                          'img_cluster': img_labels_obs}, index=common),
    )
    tmp.obs['rna_cluster'] = tmp.obs['rna_cluster'].astype('category')
    tmp.obs['img_cluster'] = tmp.obs['img_cluster'].astype('category')
    ari_observed = float(scib.metrics.ari(tmp, 'rna_cluster', 'img_cluster'))

    null_ari = np.empty(N_PERMUTATIONS_ARI, dtype=float)
    for p in range(N_PERMUTATIONS_ARI):
        tmp.obs['img_cluster'] = pd.Categorical(rng.permutation(img_labels_obs))
        null_ari[p] = float(scib.metrics.ari(tmp, 'rna_cluster', 'img_cluster'))

    p_value_ari = (np.sum(null_ari >= ari_observed) + 1) / (N_PERMUTATIONS_ARI + 1)
    print(f'\n=== confidence >= {cut} ===')
    print(f'observed ARI         = {ari_observed:.4f}')
    print(f'null  mean ± std     = {null_ari.mean():.4f} ± {null_ari.std():.4f}')
    print(f'null  max            = {null_ari.max():.4f}')
    print(f'permutation p-value  = {p_value_ari:.4g}  ({N_PERMUTATIONS_ARI} shuffles)')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(null_ari, bins=30, color='grey',
            label=f'shuffled (n={N_PERMUTATIONS_ARI})')
    ax.axvline(ari_observed, color='C3', lw=2,
               label=f'observed = {ari_observed:.3f}')
    ax.set_xlabel('Adjusted Rand Index')
    ax.set_ylabel('count')
    ax.set_title(f'ARI permutation null vs observed (conf>={cut})  one-sided p ≈ {p_value_ari:.3g}')
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_permutation_null_ari_confidence{cut}.pdf', bbox_inches='tight')
    plt.show()

    results[cut]['ari_observed'] = ari_observed
    results[cut]['null_ari'] = null_ari
    results[cut]['p_value_ari'] = p_value_ari

In [ ]:
len(rna.obs[rna.obs.llm_confidence == 5].llm_annotation.unique().tolist())

In [ ]:
# Plot examples — one page per cluster, gene names annotated on highlighted points.
from matplotlib.backends.backend_pdf import PdfPages
from adjustText import adjust_text

most_diverging = ranked.tail(30)["llm_annotation"].tolist()

pdf_path = OUTPUT_DIR / 'most_diverging_clusters_phate.pdf'
with PdfPages(pdf_path) as pdf:
    for cluster_name in most_diverging:
        genes = rna.obs_names[rna.obs["llm_annotation"].astype(str) == cluster_name].tolist()
        rna.obs["highlight"] = np.nan
        rna.obs.loc[rna.obs_names.isin(genes), "highlight"] = cluster_name
        img.obs["highlight"] = np.nan
        img.obs.loc[img.obs_names.isin(genes), "highlight"] = cluster_name

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        sc.pl.embedding(
            rna, basis=RNA_PHATE_KEY, color="highlight", ax=axes[0],
            show=False, frameon=False, na_color='darkgrey',
            legend_loc=None,
            title='RNA',
        )
        sc.pl.embedding(
            img, basis=IMG_PHATE_KEY, color='highlight', ax=axes[1],
            show=False, frameon=False, na_color='darkgrey',
            legend_loc=None,
            title='image',
        )

        # Annotate highlighted points with their gene names, repelled with adjustText.
        rna_mask = np.asarray(rna.obs_names.isin(genes))
        rna_coords = rna.obsm[RNA_PHATE_KEY][rna_mask]
        rna_genes_present = rna.obs_names[rna_mask].tolist()
        rna_texts = [axes[0].text(x, y, gene, fontsize=11, ha='center', va='bottom')
                     for (x, y), gene in zip(rna_coords, rna_genes_present)]
        adjust_text(rna_texts, ax=axes[0],
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

        img_mask = np.asarray(img.obs_names.isin(genes))
        img_coords = img.obsm[IMG_PHATE_KEY][img_mask]
        img_genes_present = img.obs_names[img_mask].tolist()
        img_texts = [axes[1].text(x, y, gene, fontsize=11, ha='center', va='bottom')
                     for (x, y), gene in zip(img_coords, img_genes_present)]
        adjust_text(img_texts, ax=axes[1],
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

        fig.suptitle(cluster_name, fontsize=11)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.show()

print(f'saved: {pdf_path}')

In [ ]:
rna.obs[rna.obs.llm_annotation == "Pol II transcription initiation: TFIID/TFIIA + SWI/SNF (GBAF/BAF)"]